# Silver Layer — Clean & Conform Professional-Services Data
Dedupe, cast types. Keeps engagement label is_over_budget; FE drops post-project leakage.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_date, row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Clean clients
df_c = spark.read.format('delta').table('bronze_clients')
w = Window.partitionBy('client_id').orderBy(col('ingestion_timestamp').desc())
df_c = (
    df_c.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('contract_value_gbp', col('contract_value_gbp').cast('double'))
    .withColumn('relationship_years', col('relationship_years').cast('int'))
    .withColumn('nps_score', col('nps_score').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_c.write.mode('overwrite').format('delta').saveAsTable('silver_clients')
print(f'silver_clients: {spark.table("silver_clients").count()} rows')

In [ ]:
# Clean consultants
df_n = spark.read.format('delta').table('bronze_consultants')
w2 = Window.partitionBy('consultant_id').orderBy(col('ingestion_timestamp').desc())
df_n = (
    df_n.withColumn('_rn', row_number().over(w2)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('daily_rate_gbp', col('daily_rate_gbp').cast('double'))
    .withColumn('years_experience', col('years_experience').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_n.write.mode('overwrite').format('delta').saveAsTable('silver_consultants')
print(f'silver_consultants: {spark.table("silver_consultants").count()} rows')

In [ ]:
# Clean engagements (keep label is_over_budget)
df_e = spark.read.format('delta').table('bronze_engagements')
w3 = Window.partitionBy('engagement_id').orderBy(col('ingestion_timestamp').desc())
df_e = (
    df_e.withColumn('_rn', row_number().over(w3)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('start_date', to_date('start_date'))
    .withColumn('planned_end_date', to_date('planned_end_date'))
    .withColumn('budget_gbp', col('budget_gbp').cast('double'))
    .withColumn('headcount', col('headcount').cast('int'))
    .withColumn('planned_duration_days', col('planned_duration_days').cast('int'))
    .withColumn('is_over_budget', col('is_over_budget').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_e.write.mode('overwrite').format('delta').saveAsTable('silver_engagements')
print(f'silver_engagements: {spark.table("silver_engagements").count()} rows')

In [ ]:
# Clean timesheets
df_t = spark.read.format('delta').table('bronze_timesheets')
df_t = (
    df_t
    .withColumn('week_starting', to_date('week_starting'))
    .withColumn('hours_logged', col('hours_logged').cast('double'))
    .withColumn('billed_value_gbp', col('billed_value_gbp').cast('double'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_t.write.mode('overwrite').format('delta').saveAsTable('silver_timesheets')
print(f'silver_timesheets: {spark.table("silver_timesheets").count()} rows')